# 🎯 Module 10: Feature Selection

> **Stage:** Machine Learning  
> **Level:** Intermediate  
> **Module:** 10 — Feature Selection  
> **Date:** 20 January 2026  
> **Previous:** 09 — Hyperparameter Tuning  
> **Next:** 11 — Time Series

---

Datasets often contain **many features** — but **not all of them help**.

**Feature Selection** picks the most relevant features and removes noisy or redundant ones.

In this notebook we'll cover:

- Why feature selection matters
- **Filter methods** — correlation, chi-square, VarianceThreshold
- **Wrapper methods** — Recursive Feature Elimination (RFE)
- **Embedded methods** — Lasso, tree importance
- The **filter vs wrapper vs embedded** framework
- **Pipeline + Cross-Validation** to avoid leakage

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain what feature selection is (and how it differs from dimensionality reduction)
- Apply **filter methods** (correlation, chi-square, VarianceThreshold)
- Apply **wrapper methods** (RFE)
- Apply **embedded methods** (Lasso, tree importance)
- Understand the **filter vs wrapper vs embedded** distinction
- Prevent **data leakage** by using a Pipeline
- **Verify** whether removing features actually helps

---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, chi2, f_classif, RFE,
)
from sklearn.metrics import accuracy_score, f1_score

np.random.seed(42)
print('✅ Setup ready.')

In [ ]:
# Load the Breast Cancer dataset as a DataFrame
data = load_breast_cancer()
X_df = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print('Shape:', X_df.shape)
print('Features:', X_df.columns.tolist()[:5], '...')
print('Classes:', data.target_names.tolist())

---
## 🧠 1. Why Feature Selection Matters

Suppose you're predicting **house prices**:

```text
Useful:
✓ Area
✓ Location
✓ Bedrooms
✓ Bathrooms

Potentially unnecessary:
✗ Random ID
✗ Duplicate information
✗ Constant column
✗ Unrelated text field
```

### Feature selection can:

- Reduce model complexity
- Improve training speed
- Reduce overfitting
- Remove noise
- Improve interpretability

### ⚠️ Feature Selection ≠ Dimensionality Reduction

| Feature Selection | Dimensionality Reduction |
| --- | --- |
| Keeps **original** features | Creates **new** transformed features |
| Subset of existing columns | Linear/nonlinear combinations |
| Examples: RFE, Lasso | Examples: PCA, UMAP |

---
## 🧹 2. Filter Methods

**Filter methods** select features using **statistical properties** — often **without training a model**.

### Common filter methods

- **Correlation** — remove redundant, highly-correlated features
- **Chi-square** — association between categorical features and target
- **VarianceThreshold** — remove features with almost no variation

### 2.1 Correlation

```text
+1 → Strong positive relationship
 0 → Little/no linear relationship
-1 → Strong negative relationship
```

If two features are highly correlated, they may carry **redundant information**.

> High correlation does **not automatically** mean one must be removed — domain knowledge matters.

In [ ]:
# Correlation heatmap
corr = X_df.corr().abs()

plt.figure(figsize=(10, 8))
plt.imshow(corr, cmap='viridis', vmin=0, vmax=1)
plt.colorbar(label='|correlation|')
plt.title('Feature correlation matrix (Breast Cancer)')
plt.xticks([]); plt.yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# Find highly correlated feature pairs (|r| > 0.95)
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr_cols = [col for col in upper.columns if any(upper[col] > 0.95)]

print(f'Features with |r| > 0.95 with another feature: {len(high_corr_cols)}')
print('First few:')
for c in high_corr_cols[:5]:
    print(f'  {c}')

### 2.2 Chi-Square Test

Chi-square (`chi2`) works when:
- Features are **categorical / non-negative**
- Target is **categorical**

It measures **association** between each feature and the target.

> For text classification, chi-square helps select **informative terms**.

In [ ]:
# Chi-square feature selection (breast cancer features are already non-negative)
selector_chi = SelectKBest(score_func=chi2, k=10)
X_chi = selector_chi.fit_transform(X_df, y)

chi_scores = pd.Series(selector_chi.scores_, index=X_df.columns).sort_values(ascending=False)
print('Top 10 features by chi-square score:')
print(chi_scores.head(10).round(2))
print(f'\nSelected shape: {X_chi.shape}')

### 2.3 VarianceThreshold

A feature with **almost no variation** provides little information.

```text
Feature A:  1, 1, 1, 1, 1, 1     ← no variance, useless
```

`VarianceThreshold` removes features whose variance is below the chosen threshold.

In [ ]:
# Add a constant column and a nearly-constant column to demonstrate
X_demo = X_df.copy()
X_demo['constant'] = 1.0
X_demo['almost_constant'] = np.concatenate([np.ones(len(X_demo) - 1), [2.0]])

vt = VarianceThreshold(threshold=0.01)
vt.fit(X_demo)

kept    = X_demo.columns[vt.get_support()].tolist()
removed = [c for c in X_demo.columns if c not in kept]

print(f'Original features: {X_demo.shape[1]}')
print(f'Kept:              {len(kept)}')
print(f'Removed:           {removed}')

---
## 🔁 3. Wrapper Methods

Wrapper methods evaluate **subsets of features by actually training a model**.

### Recursive Feature Elimination (RFE)

```text
All Features
     ↓
Train Model
     ↓
Remove Weakest
     ↓
Train Again
     ↓
Remove Weakest
     ↓
Best Feature Subset
```

### Pros / Cons

- ✅ Uses actual model performance
- ❌ Computationally expensive (trains many models)

In [ ]:
# RFE with Logistic Regression
X_train, X_test, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, random_state=42, stratify=y
)

# Scale for LR
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

rfe = RFE(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    n_features_to_select=10,
)
rfe.fit(X_train_s, y_train)

selected = X_df.columns[rfe.support_].tolist()
print(f'RFE selected {len(selected)} features:')
for f in selected:
    print(f'  - {f}')

In [ ]:
# Compare model with all features vs RFE-selected features
all_feats_lr = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_s, y_train)
rfe_feats_lr = LogisticRegression(max_iter=1000, random_state=42).fit(
    X_train_s[:, rfe.support_], y_train
)

acc_all = accuracy_score(y_test, all_feats_lr.predict(X_test_s))
acc_rfe = accuracy_score(y_test, rfe_feats_lr.predict(X_test_s[:, rfe.support_]))

print(f'All features:  {X_train_s.shape[1]:>3} → test acc = {acc_all:.4f}')
print(f'RFE selection: {rfe.support_.sum():>3} → test acc = {acc_rfe:.4f}')

---
## ⚙️ 4. Embedded Methods

Embedded methods perform feature selection **during model training**.

Common examples:

- **Lasso** (L1 regularization)
- **Decision Trees**
- **Random Forest**
- **Gradient Boosting**

### 4.1 Lasso Feature Selection

Lasso uses **L1 regularization**:

$$\text{Loss} + \lambda \sum |w_i|$$

The L1 penalty can force some coefficients to exactly **zero** — those features can be removed.

```text
Feature       Coefficient
Area              0.82
Bedrooms          0.35
Age              -0.12
Noise              0
Random_ID          0
```

In [ ]:
# Lasso for feature selection
lasso = Lasso(alpha=0.1, max_iter=10000)
lasso.fit(X_train_s, y_train)

coefs = pd.Series(lasso.coef_, index=X_df.columns)
print(f'Non-zero coefficients: {(coefs != 0).sum()} / {len(coefs)}')
print('\nTop 10 features by |coefficient|:')
print(coefs.abs().sort_values(ascending=False).head(10).round(4))

zeroed = coefs[coefs == 0].index.tolist()
print(f'\nZeroed-out features ({len(zeroed)}):')
for f in zeroed[:10]:
    print(f'  - {f}')

### 4.2 Tree-Based Feature Importance

Tree-based models like Random Forest provide **feature_importances_** — a measure of how much each feature contributes to the model's decisions.

In [ ]:
# Random Forest feature importance
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

imp = pd.Series(rf.feature_importances_, index=X_df.columns).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
imp.head(15)[::-1].plot(kind='barh')
plt.title('Top 15 feature importances (Random Forest)')
plt.xlabel('importance')
plt.tight_layout(); plt.show()

print('Top 10 features by importance:')
print(imp.head(10).round(4))

In [ ]:
# Compare: all features vs top-k by importance
for k in [5, 10, 15, 20, 30]:
    top_k = imp.head(k).index.tolist()
    rf_k = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    rf_k.fit(X_train[top_k], y_train)
    acc = accuracy_score(y_test, rf_k.predict(X_test[top_k]))
    print(f'Top {k:>2} features → test acc = {acc:.4f}')

---
## 📊 5. Filter vs Wrapper vs Embedded

| Method | Model Required? | Speed | Example |
| --- | :-: | :-: | --- |
| **Filter** | ❌ | ⚡ Fast | Correlation, Chi-square |
| **Wrapper** | ✅ | 🐢 Slow | RFE |
| **Embedded** | ✅ | ⚖️ Medium | Lasso, Trees |

### Easy way to remember

```text
Filter
→ Select BEFORE model

Wrapper
→ Model EVALUATES feature subsets

Embedded
→ Model SELECTS while learning
```

---
## 🚫 6. Feature Selection + Cross-Validation

Feature selection can cause **data leakage** if performed on the **entire dataset** before cross-validation.

### ❌ Wrong

```text
Entire Dataset
      ↓
Feature Selection
      ↓
Cross-Validation   ← test info already leaked
```

### ✅ Correct

```text
Training Data
      ↓
Cross-Validation
      ↓
Feature Selection INSIDE each fold
      ↓
Model Training
```

A **Pipeline** handles this automatically.

In [ ]:
# Pipeline with feature selection inside
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('select', SelectKBest(f_classif, k=10)),
    ('model', LogisticRegression(max_iter=1000, random_state=42)),
])

scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')

print(f'Pipeline (SelectKBest + LR) CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}')

# Compare with no selection
pipe_all = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42)),
])
scores_all = cross_val_score(pipe_all, X_train, y_train, cv=5, scoring='accuracy')
print(f'Pipeline (all features + LR)   CV accuracy: {scores_all.mean():.4f} ± {scores_all.std():.4f}')

---
## 🧪 7. Practical Workflow

```text
Dataset
   ↓
Remove IDs / Constant / Obviously Irrelevant Features
   ↓
Check Correlation
   ↓
Apply Filter / Wrapper / Embedded Method
   ↓
Cross-Validation
   ↓
Compare Performance
   ↓
Keep Useful Features
   ↓
Final Model
```

> **Don't assume fewer features = better model.** Always compare performance using proper validation.

---
## 📝 8. Practice Checklist

- [ ] Calculate feature correlations
- [ ] Remove a constant feature using `VarianceThreshold`
- [ ] Use `SelectKBest`
- [ ] Apply chi-square feature selection to a classification dataset
- [ ] Apply RFE with Logistic Regression
- [ ] Use Lasso to identify zero-coefficient features
- [ ] Extract Random Forest feature importance
- [ ] Compare model performance before and after feature selection
- [ ] Put feature selection inside a Pipeline
- [ ] Explain how feature selection can cause data leakage

---
## 🏋️ 9. Hands-On Exercises

### Exercise 1 — Correlation Threshold Sweep
Try thresholds `|r| ∈ {0.80, 0.90, 0.95, 0.99}`. How many features get removed at each threshold? Does accuracy change?

### Exercise 2 — RFE n_features Sweep
Try `n_features_to_select ∈ {5, 10, 15, 20, 25}` with LR and with Random Forest. Which model has the best trade-off?

### Exercise 3 — Lasso Alpha Sweep
Try `alpha ∈ {0.001, 0.01, 0.1, 1.0}`. Plot the number of non-zero coefficients vs alpha.

### Exercise 4 — Multi-Model Feature Comparison
Get the top-10 features from **chi-square**, **RF**, and **Lasso**. How much overlap is there? Which model benefits most?

### Exercise 5 — Leakage Demo
Run `SelectKBest` on the full dataset, then cross-validate. Compare to putting `SelectKBest` **inside** the pipeline. Which estimate is more honest?

### Exercise 6 — Feature Selection ≠ Better
Find a dataset where removing features **hurts** accuracy. What does this tell you about the workflow?

---
## 🎤 10. Interview Questions

1. **What is feature selection?**  
   Choosing the most relevant features and removing unnecessary or noisy ones.

2. **Feature selection vs dimensionality reduction?**  
   Selection keeps original features; reduction creates new transformed features.

3. **What are filter methods?**  
   Statistical measures like correlation, chi-square, VarianceThreshold — no model required.

4. **How does correlation help with feature selection?**  
   It identifies redundant features that carry similar information.

5. **What is the chi-square test used for?**  
   Measuring association between non-negative/categorical features and a categorical target.

6. **What is VarianceThreshold?**  
   Removes features whose variance is below a threshold (constant or near-constant columns).

7. **What is RFE?**  
   Recursive Feature Elimination — repeatedly trains a model, ranks features, and removes the weakest.

8. **Filter vs wrapper vs embedded?**  
   Filter — before model. Wrapper — model evaluates subsets. Embedded — model selects during training.

9. **How does Lasso perform feature selection?**  
   The L1 penalty pushes some coefficients exactly to zero.

10. **How do tree models provide feature importance?**  
    By measuring how much each feature decreases impurity across all splits in the ensemble.

11. **Can feature selection cause data leakage?**  
    Yes — if performed using test data (e.g., on the full dataset before CV).

12. **Why should feature selection be performed inside cross-validation?**  
    To keep the process honest — feature selection is part of the training process, not the evaluation.

---
## 🏁 11. Key Takeaways

- **Feature selection** keeps useful features and removes unnecessary ones
- **Filter methods** → correlation, chi-square, VarianceThreshold
- **Wrapper methods** → RFE and model-based subset evaluation
- **Embedded methods** → Lasso and tree-based importance
- **Lasso** can drive unnecessary coefficients toward zero
- **Tree models** provide feature importance
- Feature selection can improve **speed**, **interpretability**, and sometimes **generalization**
- Always **validate** whether removing features actually improves the model
- Use **Pipeline + Cross-Validation** to avoid data leakage

### 🧭 Golden Rule

> **Feature selection is a tool, not a rule. Fewer features ≠ better — validate every choice against real, honest performance.**

---

### 🔗 What's Next?

**Module 11 — Time Series**

We'll cover **ARIMA**, **SARIMA**, **Prophet**, and **LSTM basics** — the tools for forecasting sequential, time-dependent data.